## Feature Engineering Strategy

### 1. Temporal Aggregation

To capture crime patterns over time, a rolling window approach was applied.

* For each agency, crime data from the previous three years (2021–2023) was aggregated to construct features.
* The target variable was defined using crime data from 2024.
* Agencies in the top 25% of crime counts in 2024 were labeled as hotspots (label = 1), and others as non-hotspots (label = 0).

This setup simulates a real-world prediction scenario, where future crime hotspots are predicted based on historical data.

---

### 2. Offense Type Mapping and Aggregation

The original NIBRS dataset contains a large number of detailed offense categories. To reduce dimensionality and ensure consistency with the Chicago training dataset, these categories were grouped into broader crime types:

* THEFT
* BATTERY
* ASSAULT
* CRIMINAL DAMAGE
* DECEPTIVE PRACTICE
* OTHER_CRIME

Since the definition of crime categories differs between datasets, a mapping strategy was applied:

* Assault-related offenses in NIBRS (e.g., Simple Assault, Aggravated Assault) were mapped to **BATTERY**, as they involve physical contact.
* Threat-based offenses (e.g., Intimidation) were mapped to **ASSAULT**.
* Fraud-related offenses were grouped under **DECEPTIVE PRACTICE**.
* Categories without a clear match (e.g., drug-related or administrative offenses) were grouped into **OTHER_CRIME**.

---

### 3. Location Type Aggregation

Location information was also incorporated to capture environmental context.

* Detailed location types were grouped into broader categories:

  * RESIDENTIAL
  * PUBLIC_OUTDOOR
  * COMMERCIAL
  * INSTITUTION
  * OTHER

This transformation allows the model to learn how different environments influence crime patterns.


In [1]:
import pandas as pd
df_use = pd.read_csv("../data/nibrs/nibrs_use.csv",low_memory=False)

In [2]:
print(df_use.columns)

Index(['agency_id', 'year', 'offense_name', 'location_name', 'state_name',
       'region_name', 'agency_type_name'],
      dtype='object')


In [3]:
df_use["location_name"].nunique()

46

In [4]:
df_use["location_name"].value_counts().head(30)

location_name
Residence/Home                                   4134375
Highway/Road/Alley/Street/Sidewalk               2153719
Parking/Drop Lot/Garage                          1306121
Other/Unknown                                     759482
Department/Discount Store                         411497
Specialty Store                                   328184
Commercial/Office Building                        327865
Convenience Store                                 229064
Grocery/Supermarket                               229004
Hotel/Motel/Etc.                                  205211
School-Elementary/Secondary                       203904
Restaurant                                        202629
Drug Store/Doctor's Office/Hospital               154117
Service/Gas Station                               130514
Park/Playground                                   123402
Bank/Savings and Loan                              90101
Construction Site                                  85475
Bar/Nightclub    

In [5]:
loc_counts = df_use["location_name"].value_counts(normalize=True)

print(loc_counts.head(20))

location_name
Residence/Home                         0.346535
Highway/Road/Alley/Street/Sidewalk     0.180520
Parking/Drop Lot/Garage                0.109476
Other/Unknown                          0.063658
Department/Discount Store              0.034491
Specialty Store                        0.027508
Commercial/Office Building             0.027481
Convenience Store                      0.019200
Grocery/Supermarket                    0.019195
Hotel/Motel/Etc.                       0.017200
School-Elementary/Secondary            0.017091
Restaurant                             0.016984
Drug Store/Doctor's Office/Hospital    0.012918
Service/Gas Station                    0.010939
Park/Playground                        0.010343
Bank/Savings and Loan                  0.007552
Construction Site                      0.007164
Bar/Nightclub                          0.007023
Rental Storage Facility                0.007007
Shopping Mall                          0.006650
Name: proportion, dtype: f

In [6]:
#define mapping rules for location category,same as chicago dataset
def map_location(x):
    if pd.isna(x):
        return "OTHER"
    
    x = x.lower()

    # RESIDENTIAL
    if "residence" in x:
        return "RESIDENTIAL"

    # PUBLIC OUTDOOR
    elif any(k in x for k in [
        "highway", "road", "street", "sidewalk",
        "park", "parking"
    ]):
        return "PUBLIC_OUTDOOR"

    # COMMERCIAL
    elif any(k in x for k in [
        "store", "market", "restaurant", "bank",
        "hotel", "gas", "mall"
    ]):
        return "COMMERCIAL"

    # INSTITUTION
    elif any(k in x for k in [
        "school", "college", "hospital",
        "government", "church", "prison"
    ]):
        return "INSTITUTION"

    # OTHER
    else:
        return "OTHER"

In [7]:
df_use["location_category"] = df_use["location_name"].apply(map_location)

In [8]:
df_use["location_category"].unique()

array(['RESIDENTIAL', 'PUBLIC_OUTDOOR', 'OTHER', 'COMMERCIAL',
       'INSTITUTION'], dtype=object)

In [9]:
df_use.head(5)

,agency_id,year,offense_name,location_name,state_name,region_name,agency_type_name,location_category
0,1516,2021,Rape,Residence/Home,California,West,County,RESIDENTIAL
1,1516,2021,Aggravated Assault,Highway/Road/Alley/Street/Sidewalk,California,West,County,PUBLIC_OUTDOOR
2,1516,2021,Simple Assault,Residence/Home,California,West,County,RESIDENTIAL
3,1516,2021,Simple Assault,Other/Unknown,California,West,County,OTHER
4,1516,2021,Simple Assault,Convenience Store,California,West,County,COMMERCIAL


In [10]:
df_use["offense_name"].unique()

array(['Rape', 'Aggravated Assault', 'Simple Assault',
       'Stolen Property Offenses', 'Drug/Narcotic Violations',
       'Drug Equipment Violations', 'Theft From Building',
       'Weapon Law Violations', 'Shoplifting', 'All Other Larceny',
       'False Pretenses/Swindle/Confidence Game',
       'Burglary/Breaking & Entering',
       'Theft of Motor Vehicle Parts or Accessories',
       'Destruction/Damage/Vandalism of Property', 'Purse-snatching',
       'Motor Vehicle Theft', 'Statutory Rape',
       'Theft From Motor Vehicle', 'Intimidation', 'Identity Theft',
       'Robbery', 'Embezzlement', 'Arson', 'Kidnapping/Abduction',
       'Fondling', 'Counterfeiting/Forgery',
       'Credit Card/Automated Teller Machine Fraud',
       'Flight to Avoid Prosecution',
       'Theft From Coin-Operated Machine or Device', 'Impersonation',
       'Pocket-picking', 'Murder and Nonnegligent Manslaughter',
       'Extortion/Blackmail', 'Sexual Assault With An Object',
       'Harboring Escape

In [11]:
df_use["offense_name"].value_counts().head(20)

offense_name
Simple Assault                                 1666237
Destruction/Damage/Vandalism of Property       1292478
All Other Larceny                              1197974
Theft From Motor Vehicle                        884191
Drug/Narcotic Violations                        882251
Motor Vehicle Theft                             754088
Shoplifting                                     709423
Burglary/Breaking & Entering                    626223
Intimidation                                    615907
Aggravated Assault                              575927
Drug Equipment Violations                       393389
Theft of Motor Vehicle Parts or Accessories     336308
Weapon Law Violations                           281362
Theft From Building                             223757
False Pretenses/Swindle/Confidence Game         217171
Robbery                                         194975
Identity Theft                                  171495
Credit Card/Automated Teller Machine Fraud      1468

In [12]:
# The NIBRS dataset contains detailed offense categories that differ from the Chicago dataset.
# To ensure compatibility with the trained model, all offense types are mapped into a set of
# broader categories consistent with the Chicago feature schema:
# THEFT, BATTERY, ASSAULT, CRIMINAL DAMAGE, DECEPTIVE PRACTICE, and OTHER_CRIME.
#
# Note that the definition of violent crimes differs between datasets. In Chicago,
# "BATTERY" refers to physical contact, while "ASSAULT" refers to threats without contact.
# In contrast, NIBRS groups most physical attacks under assault-related categories
# (e.g., Simple Assault, Aggravated Assault). Therefore, these are mapped to "BATTERY"
# to maintain semantic consistency.
#
# Offense types that cannot be directly aligned (e.g., drug-related or administrative offenses)
# are grouped into "OTHER_CRIME".

In [13]:
def map_offense(x):
    if pd.isna(x):
        return "OTHER_CRIME"

    theft = {
        "All Other Larceny",
        "Theft From Motor Vehicle",
        "Theft From Building",
        "Theft of Motor Vehicle Parts or Accessories",
        "Motor Vehicle Theft",
        "Shoplifting",
        "Burglary/Breaking & Entering",
        "Pocket-picking",
        "Purse-snatching",
        "Theft From Coin-Operated Machine or Device",
        "Stolen Property Offenses"
    }

    battery = {
        "Simple Assault",
        "Aggravated Assault",
        "Robbery",
        "Murder and Nonnegligent Manslaughter",
        "Negligent Manslaughter",
        "Justifiable Homicide",
        "Kidnapping/Abduction",
        "Criminal Sexual Contact",
        "Fondling",
        "Rape",
        "Statutory Rape",
        "Sexual Assault With An Object",
        "Sodomy",
        "Incest"
    }

    assault = {
        "Intimidation",
        "Extortion/Blackmail"
    }

    criminal_damage = {
        "Destruction/Damage/Vandalism of Property",
        "Arson"
    }

    deceptive_practice = {
        "False Pretenses/Swindle/Confidence Game",
        "Identity Theft",
        "Counterfeiting/Forgery",
        "Credit Card/Automated Teller Machine Fraud",
        "Embezzlement",
        "Impersonation",
        "Welfare Fraud",
        "Wire Fraud",
        "Bribery"
    }

    if x in theft:
        return "THEFT"
    elif x in battery:
        return "BATTERY"
    elif x in assault:
        return "ASSAULT"
    elif x in criminal_damage:
        return "CRIMINAL DAMAGE"
    elif x in deceptive_practice:
        return "DECEPTIVE PRACTICE"
    else:
        return "OTHER_CRIME"

In [14]:
df_use["offense_group"] = df_use["offense_name"].apply(map_offense)

In [15]:
df_use["offense_group"].value_counts(normalize=True)

offense_group
THEFT                 0.410299
BATTERY               0.222285
OTHER_CRIME           0.136965
CRIMINAL DAMAGE       0.110741
DECEPTIVE PRACTICE    0.066931
ASSAULT               0.052778
Name: proportion, dtype: float64

In [16]:
df_use.head(10)

,agency_id,year,offense_name,location_name,state_name,region_name,agency_type_name,location_category,offense_group
0,1516,2021,Rape,Residence/Home,California,West,County,RESIDENTIAL,BATTERY
1,1516,2021,Aggravated Assault,Highway/Road/Alley/Street/Sidewalk,California,West,County,PUBLIC_OUTDOOR,BATTERY
2,1516,2021,Simple Assault,Residence/Home,California,West,County,RESIDENTIAL,BATTERY
3,1516,2021,Simple Assault,Other/Unknown,California,West,County,OTHER,BATTERY
4,1516,2021,Simple Assault,Convenience Store,California,West,County,COMMERCIAL,BATTERY
5,1516,2021,Simple Assault,Residence/Home,California,West,County,RESIDENTIAL,BATTERY
6,1516,2021,Stolen Property Offenses,Highway/Road/Alley/Street/Sidewalk,California,West,County,PUBLIC_OUTDOOR,THEFT
7,1516,2021,Drug/Narcotic Violations,Highway/Road/Alley/Street/Sidewalk,California,West,County,PUBLIC_OUTDOOR,OTHER_CRIME
8,1516,2021,Drug Equipment Violations,Highway/Road/Alley/Street/Sidewalk,California,West,County,PUBLIC_OUTDOOR,OTHER_CRIME
9,1516,2021,Simple Assault,Residence/Home,California,West,County,RESIDENTIAL,BATTERY


In [17]:
# Split dataset into feature years (2021–2023) and label year (2024)
df_feat = df_use[df_use["year"].isin([2021, 2022, 2023])].copy()
df_label = df_use[df_use["year"] == 2024].copy()

In [18]:
# Count number of each offense category per agency
offense_counts = pd.crosstab(df_feat["agency_id"], df_feat["offense_group"])

# Convert counts into ratios (normalize by total crimes per agency)
offense_ratio = offense_counts.div(offense_counts.sum(axis=1), axis=0)

In [19]:
# Rename columns to match the feature names used in the Chicago dataset
offense_ratio = offense_ratio.rename(columns={
    "THEFT": "theft_ratio",
    "BATTERY": "battery_ratio",
    "CRIMINAL DAMAGE": "criminal_damage_ratio",
    "ASSAULT": "assault_ratio",
    "DECEPTIVE PRACTICE": "deceptive_practice_ratio",
    "OTHER_CRIME": "other_crime_ratio"
})

In [20]:
# Ensure all expected features exist (fill missing ones with 0)
expected_cols = [
    "theft_ratio",
    "battery_ratio",
    "criminal_damage_ratio",
    "assault_ratio",
    "deceptive_practice_ratio",
    "other_crime_ratio"
]

offense_ratio = offense_ratio.reindex(columns=expected_cols, fill_value=0)

In [21]:
# Count number of each location category per agency
loc_counts = pd.crosstab(df_feat["agency_id"], df_feat["location_category"])

# Convert counts into ratios
loc_ratio = loc_counts.div(loc_counts.sum(axis=1), axis=0)

# Rename columns for clarity
loc_ratio.columns = [f"{col.lower()}_ratio" for col in loc_ratio.columns]

In [22]:
# Combine offense features and location features into one feature matrix
X = pd.concat([offense_ratio, loc_ratio], axis=1).fillna(0)

In [23]:
# Count number of crimes in 2024 per agency
label_count = df_label.groupby("agency_id").size()

# Define hotspot threshold (top 25%)
threshold = label_count.quantile(0.75)

# Assign label: 1 = hotspot, 0 = non-hotspot
y = (label_count >= threshold).astype(int)

In [24]:
# Join features and labels,keep only agencies present in both
data = X.join(y.rename("label"), how="inner")

In [25]:
print("X agencies:", len(X))
print("y agencies:", len(y))
print("final agencies:", len(data))

X agencies: 1979
y agencies: 2066
final agencies: 1914


In [26]:
# Separate features and target
X_final = data.drop(columns="label")
y_final = data["label"]

In [27]:
X_final.head(10)

,theft_ratio,battery_ratio,criminal_damage_ratio,assault_ratio,deceptive_practice_ratio,other_crime_ratio,commercial_ratio,institution_ratio,other_ratio,public_outdoor_ratio,residential_ratio
agency_id,,,,,,,,,,,
937,0.343493,0.286316,0.083018,0.013277,0.036262,0.237633,0.144122,0.180313,0.104576,0.254979,0.316011
938,0.647951,0.119794,0.097454,0.006829,0.068538,0.059433,0.171725,0.013910,0.107992,0.538105,0.168268
939,0.642501,0.087991,0.100166,0.009408,0.090205,0.069729,0.221361,0.016602,0.103486,0.500277,0.158273
941,0.728114,0.106397,0.063973,0.015657,0.048316,0.037542,0.296296,0.003704,0.045960,0.550673,0.103367
942,0.605531,0.120741,0.071727,0.009061,0.128861,0.064078,0.158046,0.015357,0.221653,0.307796,0.297146
943,0.609455,0.163379,0.123056,0.009646,0.039541,0.054923,0.130008,0.015816,0.210046,0.502651,0.141479
944,0.453997,0.157043,0.098668,0.015546,0.110406,0.164340,0.262056,0.010152,0.098985,0.410533,0.218274
945,0.623495,0.144118,0.094848,0.012037,0.054084,0.071417,0.181993,0.014604,0.092281,0.513401,0.197721
947,0.608544,0.049695,0.170881,0.020924,0.068003,0.081953,0.034002,0.017437,0.045336,0.670445,0.232781


In [28]:
y_final.head(5)

agency_id
937    1
938    1
939    0
941    1
942    1
Name: label, dtype: int32

In [29]:
X_final.reset_index().to_csv("../data/nibrs/X_test_nibrs.csv", index=False)
y_final.reset_index().to_csv("../data/nibrs/y_test_nibrs.csv", index=False)

In [32]:
# Save metadata for analysis purposes (not used for model input).
# This includes state, region, and agency type for spatial and contextual analysis.
analy_cols = ["agency_id", "state_name", "region_name", "agency_type_name"]

analy_df = df_use[analy_cols].drop_duplicates(subset="agency_id").copy()

analy_df.to_csv("../data/nibrs/nibrs_meta.csv", index=False)